In [1]:
import pandas as pd

df = pd.read_csv('Dataset_ATS_v2.csv')

print(df.shape)
print(df.dtypes)
df.head()


(7043, 10)
gender               str
SeniorCitizen      int64
Dependents           str
tenure             int64
PhoneService         str
MultipleLines        str
InternetService      str
Contract             str
MonthlyCharges     int64
Churn                str
dtype: object


,gender,SeniorCitizen,Dependents,tenure,PhoneService,MultipleLines,InternetService,Contract,MonthlyCharges,Churn
0,Female,0,No,1,No,No,DSL,Month-to-month,25,Yes
1,Male,0,No,41,Yes,No,DSL,One year,25,No
2,Female,0,Yes,52,Yes,No,DSL,Month-to-month,19,No
3,Female,0,No,1,Yes,No,DSL,One year,76,Yes
4,Male,0,No,67,Yes,No,Fiber optic,Month-to-month,51,No


In [2]:
print("Duplicate rows:", df.duplicated().sum())
print()
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Churn distribution:")
print(df['Churn'].value_counts())
print(df['Churn'].value_counts(normalize=True) * 100)


Duplicate rows: 302

Missing values per column:
gender             0
SeniorCitizen      0
Dependents         0
tenure             0
PhoneService       0
MultipleLines      0
InternetService    0
Contract           0
MonthlyCharges     0
Churn              0
dtype: int64

Churn distribution:
Churn
No     5174
Yes    1869
Name: count, dtype: int64
Churn
No     73.463013
Yes    26.536987
Name: proportion, dtype: float64


In [3]:
# Drop duplicate rows
df = df.drop_duplicates()
print("Shape after dropping duplicates:", df.shape)

# Check unique values in each categorical column before encoding
categorical_cols = ['gender', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'Contract', 'Churn']
for col in categorical_cols:
    print(col, ":", df[col].unique())
    

Shape after dropping duplicates: (6741, 10)
gender : <StringArray>
['Female', 'Male']
Length: 2, dtype: str
Dependents : <StringArray>
['No', 'Yes']
Length: 2, dtype: str
PhoneService : <StringArray>
['No', 'Yes']
Length: 2, dtype: str
MultipleLines : <StringArray>
['No', 'Yes']
Length: 2, dtype: str
InternetService : <StringArray>
['DSL', 'Fiber optic']
Length: 2, dtype: str
Contract : <StringArray>
['Month-to-month', 'One year', 'Two year']
Length: 3, dtype: str
Churn : <StringArray>
['Yes', 'No']
Length: 2, dtype: str


In [4]:
import urllib.request

url = 'https://raw.githubusercontent.com/prajwol-manandhar/telco-customer-churn-analysis/main/02_Data_Engineering/data/processed/telco_ann_ready.csv'
urllib.request.urlretrieve(url, 'telco_ann_ready.csv')

print("Downloaded successfully")


Downloaded successfully


In [5]:
import pandas as pd

df = pd.read_csv('telco_ann_ready.csv')
print(df.shape)
print(df.dtypes)
df.head()


(7043, 15)
gender                       int64
SeniorCitizen                int64
Dependents                   int64
tenure                       int64
PhoneService                 int64
MultipleLines                int64
ServiceCount                 int64
InternetService              int64
Contract_Month-to-month      int64
Contract_One year            int64
Contract_Two year            int64
ContractMonths               int64
MonthlyCharges               int64
MonthlyChargePerService    float64
Churn                        int64
dtype: object


,gender,SeniorCitizen,Dependents,tenure,PhoneService,MultipleLines,ServiceCount,InternetService,Contract_Month-to-month,Contract_One year,Contract_Two year,ContractMonths,MonthlyCharges,MonthlyChargePerService,Churn
0,0,0,0,1,0,0,0,0,1,0,0,0,25,25.0,1
1,1,0,0,41,1,0,1,0,0,1,0,12,25,12.5,0
2,0,0,1,52,1,0,1,0,1,0,0,0,19,9.5,0
3,0,0,0,1,1,0,1,0,0,1,0,12,76,38.0,1
4,1,0,0,67,1,0,1,1,1,0,0,0,51,25.5,0


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Separate features and target
X = df.drop('Churn', axis=1)
y = df['Churn']

# Split into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

# Scale the features (ANNs are sensitive to feature scale)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling done")

Training set: (5634, 14)
Testing set: (1409, 14)
Scaling done


In [7]:
from tensorflow import keras
from tensorflow.keras import layers

# Define the ANN architecture
model = keras.Sequential([
    layers.Dense(16, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    layers.Dense(8, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Matplotlib is building the font cache; this may take a moment.
/opt/anaconda3/envs/churn_env/lib/python3.11/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │           240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 385 (1.50 KB)

 Trainable params: 385 (1.50 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
!pip install tensorflow

In [9]:
!pip install tensorflow

In [10]:
import sys
print(sys.version)
print(sys.platform)


3.11.16 (main, Aug 27 2026, 14:37:43) [Clang 20.1.8 ]
darwin


In [11]:
import sys
print(sys.version)
print(sys.platform)

3.11.16 (main, Aug 27 2026, 14:37:43) [Clang 20.1.8 ]
darwin


In [12]:
!pip install numpy --upgrade

In [13]:
from tensorflow import keras
from tensorflow.keras import layers

# Define the ANN architecture
model = keras.Sequential([
    layers.Dense(16, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    layers.Dense(8, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 16)             │           240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 385 (1.50 KB)

 Trainable params: 385 (1.50 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
history = model.fit(
    X_train_scaled, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

Epoch 1/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 727us/step - accuracy: 0.5318 - loss: 0.7517 - val_accuracy: 0.7551 - val_loss: 0.5371
Epoch 2/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 335us/step - accuracy: 0.7506 - loss: 0.5080 - val_accuracy: 0.7799 - val_loss: 0.4705
Epoch 3/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 327us/step - accuracy: 0.7728 - loss: 0.4708 - val_accuracy: 0.7826 - val_loss: 0.4523
Epoch 4/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 317us/step - accuracy: 0.7819 - loss: 0.4593 - val_accuracy: 0.7799 - val_loss: 0.4454
Epoch 5/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 316us/step - accuracy: 0.7815 - loss: 0.4539 - val_accuracy: 0.7853 - val_loss: 0.4423
Epoch 6/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 414us/step - accuracy: 0.7859 - loss: 0.4508 - val_accuracy: 0.7862 - val_loss: 0.4406
Epoch 7/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 333us/step - accuracy: 0.7872 - loss: 0.4483 - val_accuracy: 0.7826 - val_loss: 0.4393
Epoch 8/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 320us/step - accuracy: 0.7894 - loss: 0.4466 - 

In [15]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

# Get predictions on the test set
y_pred_proba = model.predict(X_test_scaled)
y_pred = (y_pred_proba > 0.5).astype(int)

# Evaluate
print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("Test F1 Score:", f1_score(y_test, y_pred))
print()
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 544us/step
Test Accuracy: 0.7849538679914834
Test F1 Score: 0.5388127853881278

Classification Report:
              precision    recall  f1-score   support

    No Churn       0.83      0.90      0.86      1035
       Churn       0.63      0.47      0.54       374

    accuracy                           0.78      1409
   macro avg       0.73      0.69      0.70      1409
weighted avg       0.77      0.78      0.77      1409


Confusion Matrix:
[[929 106]
 [197 177]]


In [16]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Compute weights inversely proportional to class frequency
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))
print("Class weights:", class_weight_dict)

# Retrain with class weighting
history2 = model.fit(
    X_train_scaled, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weight_dict,
    verbose=1
)

Class weights: {0: np.float64(0.6805991785455424), 1: np.float64(1.8842809364548494)}
Epoch 1/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 470us/step - accuracy: 0.7535 - loss: 0.5168 - val_accuracy: 0.7338 - val_loss: 0.5109
Epoch 2/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 333us/step - accuracy: 0.7397 - loss: 0.5065 - val_accuracy: 0.7400 - val_loss: 0.5055
Epoch 3/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 332us/step - accuracy: 0.7448 - loss: 0.5055 - val_accuracy: 0.7356 - val_loss: 0.5103
Epoch 4/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 329us/step - accuracy: 0.7464 - loss: 0.5042 - val_accuracy: 0.7374 - val_loss: 0.5057
Epoch 5/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 329us/step - accuracy: 0.7477 - loss: 0.5032 - val_accuracy: 0.7498 - val_loss: 0.4987
Epoch 6/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 326us/step - accuracy: 0.7491 - loss: 0.5026 - val_accuracy: 0.7187 - val_loss: 0.5236
Epoch 7/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 325us/step - accuracy: 0.7455 - loss: 0.5027 - val_accuracy: 0.7329 - val_loss: 0.5109
Epoc

In [17]:
y_pred_proba = model.predict(X_test_scaled)
y_pred = (y_pred_proba > 0.5).astype(int)

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("Test F1 Score:", f1_score(y_test, y_pred))
print()
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 303us/step
Test Accuracy: 0.7154009936124911
Test F1 Score: 0.5664864864864865

Classification Report:
              precision    recall  f1-score   support

    No Churn       0.87      0.72      0.79      1035
       Churn       0.48      0.70      0.57       374

    accuracy                           0.72      1409
   macro avg       0.67      0.71      0.68      1409
weighted avg       0.76      0.72      0.73      1409


Confusion Matrix:
[[746 289]
 [112 262]]


In [18]:
from tensorflow.keras.callbacks import EarlyStopping

# Rebuild from scratch - deeper architecture with dropout
model2 = keras.Sequential([
    layers.Dense(32, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    layers.Dropout(0.3),
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(8, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model2.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model2.summary()

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history3 = model2.fit(
    X_train_scaled, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weight_dict,
    callbacks=[early_stop],
    verbose=1
)

/opt/anaconda3/envs/churn_env/lib/python3.11/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 32)             │           480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,153 (4.50 KB)

 Trainable params: 1,153 (4.50 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 787us/step - accuracy: 0.4686 - loss: 0.6796 - val_accuracy: 0.7311 - val_loss: 0.6505
Epoch 2/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 371us/step - accuracy: 0.6856 - loss: 0.6149 - val_accuracy: 0.7400 - val_loss: 0.5487
Epoch 3/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 363us/step - accuracy: 0.7100 - loss: 0.5746 - val_accuracy: 0.7303 - val_loss: 0.5317
Epoch 4/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 360us/step - accuracy: 0.7229 - loss: 0.5572 - val_accuracy: 0.7249 - val_loss: 0.5318
Epoch 5/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 360us/step - accuracy: 0.7164 - loss: 0.5533 - val_accuracy: 0.7329 - val_loss: 0.5044
Epoch 6/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 358us/step - accuracy: 0.7227 - loss: 0.5467 - val_accuracy: 0.7232 - val_loss: 0.5224
Epoch 7/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 360us/step - accuracy: 0.7253 - loss: 0.5402 - val_accuracy: 0.7276 - val_loss: 0.5131
Epoch 8/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 354us/step - accuracy: 0.7251 - loss: 0

In [19]:
y_pred_proba = model2.predict(X_test_scaled)
y_pred = (y_pred_proba > 0.5).astype(int)

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("Test F1 Score:", f1_score(y_test, y_pred))
print()
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 641us/step
Test Accuracy: 0.7224982256919801
Test F1 Score: 0.5844845908607864

Classification Report:
              precision    recall  f1-score   support

    No Churn       0.88      0.72      0.79      1035
       Churn       0.49      0.74      0.58       374

    accuracy                           0.72      1409
   macro avg       0.68      0.73      0.69      1409
weighted avg       0.78      0.72      0.74      1409


Confusion Matrix:
[[743 292]
 [ 99 275]]


In [20]:
import numpy as np
import pandas as pd

feature_names = X.columns  # from your original (unscaled) X dataframe

def get_f1(model, X_data, y_true):
    preds = (model.predict(X_data, verbose=0) > 0.5).astype(int)
    return f1_score(y_true, preds)

baseline_f1 = get_f1(model2, X_test_scaled, y_test)
print("Baseline F1:", baseline_f1)

importances = []
n_repeats = 5

for i, feature in enumerate(feature_names):
    drops = []
    for _ in range(n_repeats):
        X_permuted = X_test_scaled.copy()
        np.random.shuffle(X_permuted[:, i])  # shuffle just this one column
        permuted_f1 = get_f1(model2, X_permuted, y_test)
        drops.append(baseline_f1 - permuted_f1)
    importances.append(np.mean(drops))

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print(importance_df)

Baseline F1: 0.5844845908607864
                    Feature  Importance
3                    tenure    0.175343
12           MonthlyCharges    0.055850
13  MonthlyChargePerService    0.013841
1             SeniorCitizen    0.012276
6              ServiceCount    0.002454
7           InternetService    0.002344
4              PhoneService    0.001706
10        Contract_Two year    0.000540
8   Contract_Month-to-month   -0.000136
11           ContractMonths   -0.000802
2                Dependents   -0.001305
5             MultipleLines   -0.001705
9         Contract_One year   -0.001712
0                    gender   -0.003973


In [21]:
import joblib

# Save the trained model
model2.save('churn_ann_model.keras')

# Save the scaler (needed to preprocess new data the same way before prediction)
joblib.dump(scaler, 'churn_scaler.pkl')

# Save the feature importance results
importance_df.to_csv('feature_importance.csv', index=False)

print("Model, scaler, and feature importance all saved.")

Model, scaler, and feature importance all saved.


In [22]:
import os
print(os.listdir())

['.config', 'Music', 'churn_ann_model.keras', '.DS_Store', '.CFUserTextEncoding', '.xonshrc', '.zshrc', '.local', 'Pictures', 'feature_importance.csv', '.ipython', 'Desktop', 'Library', '.matplotlib', 'telco_ann_ready.csv', 'Public', '.tcshrc', '.anaconda', 'Movies', 'Applications', 'churn_predictive_modelling.ipynb', '.Trash', 'churn_scaler.pkl', 'Dataset_ATS_v2.csv', '.ipynb_checkpoints', '.jupyter', '.keras', 'Documents', '.bash_profile', 'Downloads', '.continuum', '.zsh_sessions', '.conda']
